In [1]:
import sys
import os
from dotenv import load_dotenv
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import metric_utils
# if notebook is in PRIN/notebooks, parent() is PRIN
#project_root = Path.cwd().resolve().parent
#sys.path.insert(0, str(project_root))
#print("Added to sys.path:", project_root)

import json
from constants import Annotations, AnnotatedReport
import time
from IPython.display import clear_output

from huggingface_hub import login

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from transformers import AutoTokenizer, DefaultDataCollator
from datasets import load_dataset, Dataset, DatasetDict

from model_utils import create_label_to_id_map, from_output_to_labels

from train_utils import get_device, add_nan_flag_to_df, create_list_of_annotated_reports, create_hugging_face_dataset, add_target_columns_to_dataset, get_normalization_stats
from classifiers import ReportExtractor
from tqdm import tqdm
import sklearn.metrics as metrics
import numpy as np
from pprint import pprint

In [2]:
############
# Parameters
############
# Data parameters
TRAIN_FILE_NAME = "train_split.csv"
VALIDATION_FILE_NAME = "validation_split.csv"
TEST_FILE_NAME = "test_split.csv"

In [3]:
plt.style.use('ggplot')

In [4]:
base_dir = Path.cwd().parent

In [5]:
# Set device
device = get_device()
print(f'{device = }')

torch.cuda.is_available() = True
torch.cuda.device_count() = 1
NVIDIA GeForce GTX 1060 6GB
device = device(type='cuda')


In [6]:
model = ReportExtractor.from_pretrained(base_dir / "saved_models" / "report_extractor", device=device)
model.to(device)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(base_dir / "saved_models" / "report_extractor")

c:\Users\lucat\PythonRepositories\PRIN\ENCODER\classifiers.py:114: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(os.path.join(load_directory, "report

In [7]:
###########
# Load data
###########
file_names = {
    'train': TRAIN_FILE_NAME,
    'validation': VALIDATION_FILE_NAME,
    'test': TEST_FILE_NAME
}
paths = {split: base_dir / "data" / file_name
         for split, file_name in file_names.items()}
data = {split: pd.read_csv(path) for split, path in paths.items()}
train_data, validation_data, test_data = data['train'], data['validation'], data['test']
# Log
print(f"{len(train_data) = }")
print(f"{len(validation_data) = }")
print(f"{len(test_data) = }")
# Create nan-flag columns
for split, df in data.items():
    add_nan_flag_to_df(df)
# Create lists of Annotated reports
annotated_reports =  {split: create_list_of_annotated_reports(data[split]) for split in file_names}

len(train_data) = 167
len(validation_data) = 46
len(test_data) = 56


In [8]:
# Check the maximum number of tokens for each report
print('model context length:', model.encoder.config.max_position_embeddings)
for split in annotated_reports:
    print(f'{split}: {len(annotated_reports[split])} reports')
    max_n_tokens = 0
    del_list = []
    for i, report in enumerate(annotated_reports[split]):
        x = tokenizer(report.report_text, return_tensors='pt')['input_ids'].shape[1]
        max_n_tokens = max(max_n_tokens, x)
        if x > model.encoder.config.max_position_embeddings:
            del_list.append(i)
    print(del_list)
    print(f'{max_n_tokens = }')
    # Delete long reports
    for i in del_list[::-1]:
        annotated_reports[split].pop(i)
    print(f'After deletion: {len(annotated_reports[split])} reports')

Token indices sequence length is longer than the specified maximum sequence length for this model (1427 > 512). Running this sequence through the model will result in indexing errors


model context length: 512
train: 167 reports
[0, 2, 10, 25, 30, 33, 38, 77, 91, 93, 97, 98, 99, 115, 116, 127, 130, 134, 148, 149, 150, 158, 161, 162, 163, 164, 166]
max_n_tokens = 1427
After deletion: 140 reports
validation: 46 reports
[7, 11, 23, 33, 34, 37, 38, 42, 44]
max_n_tokens = 1008
After deletion: 37 reports
test: 56 reports
[2, 6, 16, 23, 26, 28, 33, 37, 42, 43, 45, 47]
max_n_tokens = 861
After deletion: 44 reports


In [9]:
model.normalization_stats

{'ore_inizio': (np.float64(9.575), np.float64(3.247210341200582)),
 'ore_fine': (np.float64(10.4), np.float64(3.3674916480965473)),
 'spessore_parietale': (np.float64(19.833333333333332),
  np.float64(12.733376963276038)),
 'estensione_cranio_caudale': (np.float64(48.18320610687023),
  np.float64(16.8340809921614)),
 'distanza_oai': (np.float64(43.3968253968254), np.float64(27.65693442281567)),
 'linfonodi_sospetti': (np.float64(0.9854014598540146),
  np.float64(1.4037769774197164)),
 'numero_depositi': (np.float64(0.05), np.float64(0.2485673234472188))}

In [10]:
##############################
# Create Hugging Face Datasets
##############################
dataset = DatasetDict({
    split: create_hugging_face_dataset(annotated_reports[split])
    for split in annotated_reports
})
# Tokenize text and set format to torch
def tokenize_function(examples):
    return tokenizer(examples['text'], padding="longest", max_length=model.encoder.config.max_position_embeddings)
dataset = dataset.map(tokenize_function, batched=True)
cols_to_remove = [col for col in ["token_type_ids"] if col in dataset['validation'].column_names]
dataset = dataset.remove_columns(cols_to_remove)
dataset.set_format('torch')
# Log before adding columns
print(dataset)
# Add annotation fields to the dataset
label_to_id_map = create_label_to_id_map(model.annotations_model)
for split, reports in annotated_reports.items():
    dataset[split] = add_target_columns_to_dataset(dataset=dataset[split],
                                                   annotated_reports=reports,
                                                   label_to_id_map=label_to_id_map,
                                                   classification_columns=model.classification_fields,
                                                   binary_classification_columns=model.binary_classification_fields,
                                                   multiple_choice_columns=model.multiple_choice_fields,
                                                   regression_columns=model.regression_fields,
                                                   normalization_stats=model.normalization_stats)
# Log after adding columns
print(dataset)

Map:   0%|          | 0/140 [00:00<?, ? examples/s]

Map:   0%|          | 0/37 [00:00<?, ? examples/s]

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'input_ids', 'attention_mask'],
        num_rows: 140
    })
    validation: Dataset({
        features: ['text', 'input_ids', 'attention_mask'],
        num_rows: 37
    })
    test: Dataset({
        features: ['text', 'input_ids', 'attention_mask'],
        num_rows: 44
    })
})
DatasetDict({
    train: Dataset({
        features: ['text', 'input_ids', 'attention_mask', 'morfologia', 'riflessione_peritoneale_anteriore', 'infiltrazione_tessuto_adiposo', 'infiltrazione_sfinteri', 'infiltrazione_organi_extra', 'coinvolgimento_riflessione_peritoneale', 'coinvolgimento_fascia_mesorettale', 'depositi_tumorali', 'emvi_esteso', 'stadio_T', 'stadio_N', 'mrf', 'emvi', 'metastasi', 'ore_inizio_is_nan', 'ore_fine_is_nan', 'spessore_parietale_is_nan', 'estensione_cranio_caudale_is_nan', 'distanza_oai_is_nan', 'numero_linfonodi_non_conosciuto', 'stadio_N1c', 'ore_inizio', 'ore_fine', 'spessore_parietale', 'estensione_cranio_caudale', 

In [11]:
input_ids = dataset['test']['input_ids'].to(device)
attention_mask = dataset['test']['attention_mask'].to(device)

In [12]:
with torch.no_grad():
    output = model(input_ids=input_ids, attention_mask=attention_mask)

In [13]:
x = from_output_to_labels(model_output=output,
                      regression_fields=model.regression_fields,
                      binary_fields=model.binary_classification_fields,
                      classification_fields=model.classification_fields,
                      multiple_choice_fields=model.multiple_choice_fields,
                      label_to_id_map=label_to_id_map,
                      normalization_stats=model.normalization_stats)

c:\Users\lucat\PythonRepositories\PRIN\ENCODER\model_utils.py:176: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  tensor = torch.nn.functional.softmax(model_output[field])


In [20]:
campo = 'posizione'

In [21]:
print(output[campo][:5])
print(x[campo][:5])
print(dataset['test'][campo][:5])

tensor([[-0.3329,  0.0990, -0.5769, -2.7535],
        [-0.4450,  0.1237, -0.3940, -2.5280],
        [-0.2910,  0.0481, -0.4897, -2.9307],
        [-0.3651,  0.1571, -0.4718, -2.8592],
        [-0.4531, -0.1245, -0.4807, -2.5491]], device='cuda:0')
tensor([[0.4175, 0.5247, 0.3597, 0.0599],
        [0.3905, 0.5309, 0.4027, 0.0739],
        [0.4277, 0.5120, 0.3800, 0.0507],
        [0.4097, 0.5392, 0.3842, 0.0542],
        [0.3886, 0.4689, 0.3821, 0.0725]])
tensor([[1, 1, 0, 0],
        [0, 1, 1, 0],
        [1, 0, 0, 0],
        [1, 0, 0, 0],
        [1, 1, 0, 0]])


In [29]:
(x[campo][:5].numpy() > np.array((0.5, 0.52, 0.5, 0.5))).astype(int)

array([[0, 1, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 0, 0]])

In [16]:
label_to_id_map[campo]['id_to_label']

{0: 'solido_polipoide', 1: 'solido_anulare', 2: 'mucinoso', 3: 'NaN'}

In [17]:
np.array((1, 2, 3, 4)).argmax(axis=-1)

np.int64(3)

In [34]:
def metrics_multilabel(y_true, pred_prob, id_to_label_dict: dict[int, str],
                       thresholds: float | list[float] = 0.5, plot=False, field_name=None):

    # --- Predizioni ---
    if type(thresholds) is list:
        thresholds = np.array(thresholds)
    y_pred = (pred_prob > thresholds).astype(int)

    labels = list(id_to_label_dict.values())

    # --- Metriche principali ---
    report = metrics.classification_report(
        y_true, y_pred,
        target_names=labels,
        output_dict=True,
        zero_division=0.0
    )

    # Cohen’s kappa multilabel
    kappa = metrics.cohen_kappa_score(
        y_true.argmax(axis=1),
        y_pred.argmax(axis=1)
    )
    report["cohen_kappa"] = kappa

    # --- Plot ---
    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(20, 5))

        if field_name is not None:
            fig.suptitle(field_name, fontsize="xx-large")

        # 1. Confusion matrix per classe (multilabel → una per classe)
        metrics.ConfusionMatrixDisplay.from_predictions(
            y_true.argmax(axis=1),
            y_pred.argmax(axis=1),
            ax=axes[0],
            xticks_rotation=45,
            labels=labels
        )
        axes[0].grid(False)
        axes[0].set_title("Confusion Matrix (argmax)")

        # 2. Barplot F1 per classe
        f1_per_class = metrics.f1_score(
            y_true, y_pred, average=None, zero_division=0.0
        )
        axes[1].bar(labels, f1_per_class)
        axes[1].set_title("F1 per classe")
        axes[1].set_ylabel("F1 score")
        axes[1].tick_params(axis='x', rotation=45)

        plt.tight_layout()
        plt.show()

    return report


In [35]:
report = metrics_multilabel(dataset['test'][campo].numpy(), x[campo].numpy(),  id_to_label_dict=label_to_id_map[campo]['id_to_label'], field_name=campo)

In [36]:
pprint(report)

{'alto': {'f1-score': 0.0, 'precision': 0.0, 'recall': 0.0, 'support': 12.0},
 'basso': {'f1-score': 0.0, 'precision': 0.0, 'recall': 0.0, 'support': 20.0},
 'cohen_kappa': -0.02666666666666684,
 'giunzione': {'f1-score': 0.0,
               'precision': 0.0,
               'recall': 0.0,
               'support': 1.0},
 'macro avg': {'f1-score': 0.1532258064516129,
               'precision': 0.11875,
               'recall': 0.2159090909090909,
               'support': 55.0},
 'medio': {'f1-score': 0.6129032258064516,
           'precision': 0.475,
           'recall': 0.8636363636363636,
           'support': 22.0},
 'micro avg': {'f1-score': 0.4,
               'precision': 0.475,
               'recall': 0.34545454545454546,
               'support': 55.0},
 'samples avg': {'f1-score': 0.3409090909090909,
                 'precision': 0.4318181818181818,
                 'recall': 0.29545454545454547,
                 'support': 55.0},
 'weighted avg': {'f1-score': 0.245161290322